# MMTFv3 CL+GC ? PPO on V3Continuous Environment

Converts the `V3ContinuousPrep` sample pipeline into a Gymnasium RL environment and
trains a **PPO** agent with a custom **BaseFeaturesExtractor**.

## RL Stack
1. `V3ContinuousPrep.build_samples(...)` -> bar-level multi-modal samples
2. `V3ContinuousPPOEnv` -> Gymnasium environment over those samples
3. `V3ContinuousExtractor` -> SB3 `BaseFeaturesExtractor` for V3 modalities
4. `PPO("MultiInputPolicy", ...)` -> actor-critic agent

## Action and Reward
- Action space: `0=Short (-1), 1=Flat (0), 2=Long (+1)`
- Reward: `(prev_position_ticker * forward_return - tc_cost * |delta_position|) * reward_scale`
- Position state is tracked independently for each ticker.



## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Configuration

In [ ]:
# --- Tickers ---
TICKERS = ['CL', 'GC']

# --- Architecture ---
TUNE_BACKBONE = False     # Let Optuna choose mamba/transformer
BACKBONE = 'mamba'       # Default if TUNE_BACKBONE=False

# --- Target ---
TARGET_HORIZON_MINUTES = 30   # 30min forward return
BAR_MINUTES = 5               # 5min bars → target = y_fwd_6

# --- Session Filter ---
# Options:
#   SAMPLE_SESSION = "usa"                     # Named: "usa", "london", "overlap"
#   SAMPLE_SESSION = None                      # Use all active session bars
#   SAMPLE_SESSION_START/END = "09:30"/"15:00" # Custom time window
SAMPLE_SESSION = "usa"              # Filter to USA session only
SAMPLE_SESSION_START = None         # Custom start (overrides SAMPLE_SESSION if both set)
SAMPLE_SESSION_END = None           # Custom end

# --- Stride ---
# stride=6 with 5min bars + 30min target → non-overlapping samples
# stride=3 → 50% overlap, stride=1 → every bar (max overlap)
SAMPLE_STRIDE = 6

# --- AE ---
AE_WINDOW = 21                # 21 trading days lookback for regime VAE
F_AE = 4                      # [ret_1d, ret_5d, ret_21d, rv_1d]

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'mmtfv3_cl_gc'
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_cl_gc')

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Tickers: {TICKERS}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return")
if SAMPLE_SESSION_START and SAMPLE_SESSION_END:
    print(f"Session filter: custom {SAMPLE_SESSION_START}-{SAMPLE_SESSION_END}")
elif SAMPLE_SESSION:
    print(f"Session filter: {SAMPLE_SESSION.upper()}")
else:
    print(f"Session filter: all active bars")
print(f"Stride: {SAMPLE_STRIDE} bars ({SAMPLE_STRIDE * BAR_MINUTES}min between samples)")
print(f"AE window: {AE_WINDOW} days, f_ae={F_AE}")
if TUNE_BACKBONE:
    print(f"Backbone: Optuna-tuned (mamba / transformer)")
else:
    print(f"Backbone: {BACKBONE.upper()} (fixed)")

In [ ]:
# Verify data files for each ticker
print("Checking data files...")
required_files = ['intraday.csv', 'profiles.npz', 'rasterized.npz', 'vpin.parquet']

all_found = True
for ticker in TICKERS:
    ticker_path = DATA_ROOT / ticker
    print(f"\n{ticker}:")
    for fname in required_files:
        fpath = ticker_path / fname
        status = "[OK]" if fpath.exists() else "[MISSING]"
        print(f"  {status} {fname}")
        if not fpath.exists():
            all_found = False

if not all_found:
    print("\n[WARNING] Some files missing - data loading may fail")

## 3. Load Data via V3ContinuousPrep

In [ ]:
from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    v3_collate_fn,
    unpack_v3_batch,
    build_v3_loaders,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec

print("Loading ticker data...")
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec("USA", "08:30", "16:00")],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)

dims = prep.get_dims()
print(f"\nFeature dimensions: {dims}")
print(f"Tech feature columns ({dims['f_tech']}): {prep._tech_feature_cols[:10]}...")
print(f"n_tickers: {prep.n_tickers}")
print(f"n_asset_classes: {prep.n_asset_classes}")
print(f"n_asset_subclasses: {prep.n_asset_subclasses}")

In [ ]:
# Verify tech features include tod_sin
assert 'tod_sin' in prep._tech_feature_cols, "tod_sin missing from feature cols!"
print("tod_sin confirmed in feature columns")

# Quick target distribution check
fig, axes = plt.subplots(1, len(TICKERS), figsize=(6*len(TICKERS), 4))
if len(TICKERS) == 1:
    axes = [axes]

for ticker, ax in zip(TICKERS, axes):
    df = prep._tech_dfs.get(ticker)
    if df is None:
        continue
    target = df[prep._target_col].dropna()
    ax.hist(target.values, bins=100, alpha=0.7, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{ticker} Target ({prep._target_col})')
    ax.set_xlabel('30min Forward Log Return')
    ax.set_ylabel('Count')
    print(f"{ticker}: mean={target.mean():.6f}, std={target.std():.6f}, n={len(target)}")

plt.tight_layout()
plt.show()

## 4. Build RL Environments from V3ContinuousPrep


In [ ]:
from stable_baselines3 import PPO

from CTAFlow.models.deep_learning.rl import (
    V3ContinuousPPOEnv,
    build_v3_rl_envs,
    V3ContinuousExtractor,
)

print('RL modules imported (env + extractor + PPO)')



In [ ]:
# RL environment configuration
TECH_LOOKBACK = 64
SEQ_LOOKBACK = 12
VAL_RATIO = 0.2
VAL_CUTOFF_DATE = None

TRANSACTION_COST_BPS = 1.0
REWARD_SCALE = 100.0
MAX_EPISODE_STEPS = None

train_env, val_env, env_info = build_v3_rl_envs(
    prep,
    tech_lookback=TECH_LOOKBACK,
    seq_lookback_bars=SEQ_LOOKBACK,
    val_ratio=VAL_RATIO,
    val_cutoff_date=VAL_CUTOFF_DATE,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    reward_scale=REWARD_SCALE,
    max_episode_steps=MAX_EPISODE_STEPS,
)

print('Built V3 RL environments')
print(f"  train_samples: {env_info['train_samples']}")
print(f"  val_samples:   {env_info['val_samples']}")
print(f"  cutoff_date:   {env_info['cutoff_date']}")
print(f"  n_tickers:     {env_info['n_tickers']}")
print(f"  obs_shapes:    {env_info['train_obs_shapes']}")



## 5. Configure and Train PPO


In [ ]:
# PPO / policy configuration
PPO_CFG = {
    'learning_rate': 3e-4,
    'n_steps': 2048,
    'batch_size': 256,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.01,
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
}

EXTRACTOR_CFG = {
    'd_model': 128,
    'tech_hidden': 128,
    'id_emb_dim': 16,
    'dropout': 0.1,
}

policy_kwargs = dict(
    features_extractor_class=V3ContinuousExtractor,
    features_extractor_kwargs=EXTRACTOR_CFG,
    net_arch=dict(pi=[256, 128], vf=[256, 128]),
)

ppo_device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PPO(
    policy='MultiInputPolicy',
    env=train_env,
    policy_kwargs=policy_kwargs,
    learning_rate=PPO_CFG['learning_rate'],
    n_steps=PPO_CFG['n_steps'],
    batch_size=PPO_CFG['batch_size'],
    n_epochs=PPO_CFG['n_epochs'],
    gamma=PPO_CFG['gamma'],
    gae_lambda=PPO_CFG['gae_lambda'],
    clip_range=PPO_CFG['clip_range'],
    ent_coef=PPO_CFG['ent_coef'],
    vf_coef=PPO_CFG['vf_coef'],
    max_grad_norm=PPO_CFG['max_grad_norm'],
    device=ppo_device,
    verbose=1,
)

print(f'PPO model initialized on {ppo_device}')



In [ ]:
TOTAL_TIMESTEPS = 300_000
print(f'Training PPO for {TOTAL_TIMESTEPS:,} timesteps...')
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print('PPO training complete')



In [ ]:
def rollout_policy(env, model, deterministic=True, max_steps=None):
    obs, reset_info = env.reset()
    done = False
    step = 0
    rows = []

    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        action = int(action)
        obs, reward, terminated, truncated, info = env.step(action)

        rows.append({
            'step': step,
            'action': action,
            'reward': float(reward),
            'ticker': info.get('ticker'),
            'ticker_id': int(info.get('ticker_id', 0)),
            'date': str(info.get('date')),
            'position_prev': float(info.get('position_prev', 0.0)),
            'position_target': float(info.get('position_target', 0.0)),
            'step_return': float(info.get('step_return', 0.0)),
            'gross_pnl': float(info.get('gross_pnl', 0.0)),
            'cost': float(info.get('cost', 0.0)),
        })

        step += 1
        done = bool(terminated or truncated)
        if max_steps is not None and step >= max_steps:
            done = True

    df = pd.DataFrame(rows)
    if df.empty:
        metrics = {
            'n_steps': 0,
            'reward_mean': 0.0,
            'reward_std': 0.0,
            'sharpe': 0.0,
            'sortino': 0.0,
            'win_rate': 0.0,
            'profit_factor': 0.0,
            'avg_exposure': 0.0,
            'dir_accuracy': 0.0,
            'max_drawdown': 0.0,
        }
        return metrics, df

    rewards = df['reward'].to_numpy(dtype=float)
    reward_mean = float(np.mean(rewards))
    reward_std = float(np.std(rewards) + 1e-8)
    sharpe = reward_mean / reward_std

    downside = rewards[rewards < 0]
    downside_std = float(np.sqrt(np.mean(np.square(downside))) + 1e-8) if len(downside) else 1e-8
    sortino = reward_mean / downside_std

    win_rate = float(np.mean(rewards > 0) * 100.0)
    gross_profit = float(np.sum(rewards[rewards > 0]))
    gross_loss = float(np.abs(np.sum(rewards[rewards < 0])) + 1e-8)
    profit_factor = gross_profit / gross_loss

    avg_exposure = float(np.mean(np.abs(df['position_target'].to_numpy(dtype=float))))

    pos_prev = df['position_prev'].to_numpy(dtype=float)
    step_ret = df['step_return'].to_numpy(dtype=float)
    non_flat = np.abs(pos_prev) > 1e-6
    if np.any(non_flat):
        dir_hits = (np.sign(pos_prev[non_flat]) == np.sign(step_ret[non_flat]))
        dir_accuracy = float(np.mean(dir_hits) * 100.0)
    else:
        dir_accuracy = 0.0

    cum = np.cumsum(rewards)
    running_max = np.maximum.accumulate(cum)
    max_drawdown = float(np.max(running_max - cum))

    metrics = {
        'n_steps': int(len(df)),
        'reward_mean': reward_mean,
        'reward_std': reward_std,
        'sharpe': float(sharpe),
        'sortino': float(sortino),
        'win_rate': win_rate,
        'profit_factor': float(profit_factor),
        'avg_exposure': avg_exposure,
        'dir_accuracy': dir_accuracy,
        'max_drawdown': max_drawdown,
    }
    return metrics, df



In [ ]:
train_metrics, train_rollout = rollout_policy(train_env, model, deterministic=True)
val_metrics, val_rollout = rollout_policy(val_env, model, deterministic=True)

print('Train metrics:')
for k, v in train_metrics.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

print('\nValidation metrics:')
for k, v in val_metrics.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')



## 6. RL Visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Validation reward distribution
ax = axes[0]
ax.hist(val_rollout['reward'].values, bins=80, alpha=0.7, color='steelblue')
ax.axvline(x=0.0, color='red', linestyle='--', alpha=0.6)
ax.set_title('Validation Step Reward Distribution')
ax.set_xlabel('Reward')
ax.set_ylabel('Count')
ax.grid(True, alpha=0.3)

# Cumulative reward
ax = axes[1]
cum_reward = np.cumsum(val_rollout['reward'].values)
ax.plot(cum_reward, color='navy', alpha=0.9)
ax.axhline(0.0, color='gray', linestyle=':')
ax.set_title('Validation Cumulative Reward')
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Reward')
ax.grid(True, alpha=0.3)

# Position exposure
ax = axes[2]
ax.plot(np.abs(val_rollout['position_target'].values), color='darkgreen', alpha=0.85)
ax.set_title('Validation Absolute Exposure')
ax.set_xlabel('Step')
ax.set_ylabel('|Position|')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'ppo_v3_validation_overview.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# Action distribution by ticker (validation)
action_pivot = (
    val_rollout
    .groupby(['ticker', 'action'])
    .size()
    .unstack(fill_value=0)
)

print('Validation action counts by ticker (0=Short,1=Flat,2=Long):')
print(action_pivot)



## 7. Environment and Policy Diagnostics


In [ ]:
print('Environment info:')
for k, v in env_info.items():
    print(f'  {k}: {v}')



In [ ]:
sample_obs, _ = val_env.reset()
print('Sample observation shapes from validation env:')
for k, v in sample_obs.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: {v.shape}')
    else:
        print(f'  {k}: scalar')



In [ ]:
print('PPO policy:')
print(model.policy)



In [ ]:
# Placeholder for additional PPO diagnostics
print('Training timesteps completed:', TOTAL_TIMESTEPS)



In [ ]:
# PPO episode reward snapshot (if available)
if hasattr(model, 'ep_info_buffer') and len(model.ep_info_buffer) > 0:
    ep_rewards = [x['r'] for x in model.ep_info_buffer if 'r' in x]
    ep_lengths = [x['l'] for x in model.ep_info_buffer if 'l' in x]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ep_rewards, color='purple', alpha=0.8)
    axes[0].set_title('Episode Rewards (recent buffer)')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(ep_lengths, color='teal', alpha=0.8)
    axes[1].set_title('Episode Lengths (recent buffer)')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Length')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_PATH / 'ppo_v3_episode_buffer.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No ep_info_buffer data available.')



## 8. Save PPO Artifacts


In [ ]:
prefix = f"v3ppo_{'_'.join(TICKERS).lower()}"
model_path = RESULTS_PATH / f"{prefix}_model"
metrics_path = RESULTS_PATH / f"{prefix}_metrics.json"
train_rollout_path = RESULTS_PATH / f"{prefix}_train_rollout.csv"
val_rollout_path = RESULTS_PATH / f"{prefix}_val_rollout.csv"



In [ ]:
# Save rollouts
train_rollout.to_csv(train_rollout_path, index=False)
val_rollout.to_csv(val_rollout_path, index=False)

# Save quick cumulative reward figure
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(np.cumsum(val_rollout['reward'].values), label='Validation', color='navy')
ax.plot(np.cumsum(train_rollout['reward'].values), label='Train', color='gray', alpha=0.7)
ax.set_title('Cumulative Reward: Train vs Validation')
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_cumulative_reward.png", dpi=150, bbox_inches='tight')
plt.show()



## 9. Final Save and Summary


In [ ]:
# Save PPO model (.zip)
model.save(str(model_path))

artifact = {
    'architecture': 'PPO(MultiInputPolicy) + V3ContinuousExtractor',
    'tickers': TICKERS,
    'target_horizon_minutes': TARGET_HORIZON_MINUTES,
    'tech_lookback': TECH_LOOKBACK,
    'seq_lookback': SEQ_LOOKBACK,
    'sample_stride': SAMPLE_STRIDE,
    'session_filter': {
        'sample_session': SAMPLE_SESSION,
        'sample_session_start': SAMPLE_SESSION_START,
        'sample_session_end': SAMPLE_SESSION_END,
    },
    'env_info': env_info,
    'ppo_config': PPO_CFG,
    'extractor_config': EXTRACTOR_CFG,
    'total_timesteps': TOTAL_TIMESTEPS,
    'train_metrics': train_metrics,
    'val_metrics': val_metrics,
    'paths': {
        'model': str(model_path) + '.zip',
        'metrics_json': str(metrics_path),
        'train_rollout_csv': str(train_rollout_path),
        'val_rollout_csv': str(val_rollout_path),
    },
}

with open(metrics_path, 'w') as f:
    json.dump(artifact, f, indent=2, default=str)

print('\n' + '=' * 60)
print('PPO TRAINING COMPLETE')
print('=' * 60)
print(f"Model: {model_path}.zip")
print(f"Validation Sharpe: {val_metrics['sharpe']:.6f}")
print(f"Validation Sortino: {val_metrics['sortino']:.6f}")
print(f"Validation Win Rate: {val_metrics['win_rate']:.2f}%")
print(f"Validation Avg Exposure: {val_metrics['avg_exposure']:.4f}")
print(f"\nArtifacts saved to: {RESULTS_PATH}")

